# Sources and gas heating

Two extensions of the global model of the [first tutorial](/wellbalanced/tutorials/global-model/):
external volume sources (`VolumeSource`), such as the electrons and ions a fast beam
makes, and the gas temperature as a balance of heating by the reactions against
conduction to the walls (`GasTemperature`). The argon data are the same illustrative
step cross section.

In [ ]:
from plasmacoll import CrossSection
from plasmacoll.constants import ELECTRON_MASS

from wellbalanced import (
    ELECTRON,
    Chamber,
    GasTemperature,
    GlobalModel,
    Reaction,
    Species,
    VolumeSource,
)

argon = Species.atomic("Ar", 39.948)
argon_ion = Species.atomic("Ar+", 39.948, 1)
ionization = Reaction.from_cross_section(
    ("e", "Ar"),
    ("e", "e", "Ar+"),
    CrossSection.constant(2.0e-20, threshold=15.76),
    ELECTRON_MASS,
    electron_energy_loss=15.76,
    name="ionization",
)
chamber = Chamber(volume=1.0e-3, wall_area=0.06, h_factor=0.5)


def argon_model(**settings):
    defaults = {
        "species": [ELECTRON, argon, argon_ion],
        "reactions": [ionization],
        "chamber": chamber,
        "power": 100.0,
        "fixed": {"Ar": 1.0e20},
    }
    return GlobalModel(**{**defaults, **settings})

## 1. A beam as the only source

A `VolumeSource` adds `rate * prod(n_per)` events per m³ and s, each changing the
densities by `changes` and giving the electrons `electron_energy` (eV). Here a beam strips
electrons at a fixed rate, 300 eV each, and ionizes the gas at a rate per gas atom; no
power is absorbed otherwise.

In [ ]:
stripping = VolumeSource({"e": 1}, 1.0e19, electron_energy=300.0, name="stripping")
pairs = VolumeSource({"e": 1, "Ar+": 1}, 1.0e-2, per=("Ar",), name="beam ionization")
beam = argon_model(power=0.0, sources=[stripping, pairs]).steady_state()
beam.converged, beam.electron_temperature, beam.electron_density

The stripped electrons are a net negative charge: more electrons than ions reach the walls,
and the sheath is lower. The power terms still sum to zero.

In [ ]:
beam.wall_losses["e"] - beam.wall_losses["Ar+"], beam.power

A source's rate may also be a function of the temperatures, `rate(T_e, T_g)`, for a source
that follows the plasma (sputtering by the ion flux to a target, say), or an array on the
cells in a `Fluid1D`.

## 2. The gas temperature

With `gas_temperature=GasTemperature(...)` the gas temperature is an unknown: the heating
of the reactions (their `gas_heating` per event, in eV) against the conduction
$\kappa (T_g - T_w) / \Lambda^2$. A reaction that heats the gas by 0.5 eV per event:

In [ ]:
heating = Reaction(
    ("e", "Ar"),
    ("e", "Ar"),
    1e-14,
    electron_energy_loss=0.5,
    gas_heating=0.5,
    name="gas heating collisions",
)
gas = GasTemperature(wall_temperature=0.025, conductivity=0.02, diffusion_length=0.02)
heated = argon_model(reactions=[ionization, heating], gas_temperature=gas)
state = heated.steady_state()
state.converged, state.gas_temperature

In [ ]:
state.power["gas heating"], state.power["gas conduction"]

The gas temperature rises with the absorbed power:

In [ ]:
import plasma_plots

from wellbalanced import stack

powers = [50.0, 100.0, 200.0, 400.0, 800.0]
scan = stack(heated.scan("power", powers), "power", powers, units="W")
with plasma_plots.figure(1, 2, figsize=(9, 3.6)) as fig:
    scan.gas_temperature.plasma.plot.lineout(ax=fig[0])
    scan.density.sel(species="e").plasma.plot.lineout(ax=fig[1])